In [1]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

d:\AI\ai_env\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [2]:

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [3]:
for i in range(5):
    print(dataset["train"][i]["text"])
    print(dataset["train"][i]["label"])
    print("-"*50)

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

In [5]:
# counting positive and negative reviews
positive_count = 0
negative_count = 0
for i in range(len(dataset["train"])):
    if dataset["train"][i]["label"] == 1:
        positive_count += 1
    else:
        negative_count += 1

print(f"Positive reviews: {positive_count}")
print(f"Negative reviews: {negative_count}")

Positive reviews: 12500
Negative reviews: 12500


In [6]:
max_length = 0
min_length = float('inf')
avg_length = 0
for i in range(len(dataset["train"])):
    review_length = len(dataset["train"][i]["text"].split())
    avg_length += review_length
    if review_length > max_length:
        max_length = review_length
    if review_length < min_length:
        min_length = review_length

avg_length /= len(dataset["train"])

print(f"Max review length: {max_length}")
print(f"Min review length: {min_length}")
print(f"Average review length: {avg_length}")

Max review length: 2470
Min review length: 10
Average review length: 233.7872


In [8]:
PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"

vocab={
    PAD_TOKEN: 0,
    SOS_TOKEN: 1,
    EOS_TOKEN: 2,
    UNK_TOKEN: 3
}

for i in range(len(dataset["train"])):
    words = dataset["train"][i]["text"].split()
    for word in words:
        if word not in vocab:
            vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 280621


In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [15]:
# tokenize the first 5 reviews
print("Tokenizing the first 5 reviews using the BERT tokenizer:")
for i in range(5):
    tokens = tokenizer.tokenize(dataset["train"][i]["text"], padding="max_length", truncation=True, max_length=512)
    print(tokens)
    print("-"*50)   

# print("Encoding the first 5 reviews using the custom vocabulary:")
# for i in range(5):
#     tokens=dataset["train"][i]["text"].split()
#     encoded = [vocab[SOS_TOKEN]]
#     for token in tokens:
#         if token in vocab:
#             encoded.append(vocab[token])
#         else:
#             encoded.append(vocab[UNK_TOKEN])
#     print(encoded)
#     print("-"*50)

for i in range(5):
    words = dataset["train"][i]["text"].split()
    print(f"Review {i+1}:")
    print(f"Words: {words}")

Tokenizing the first 5 reviews using the BERT tokenizer:
['i', 'rented', 'i', 'am', 'curious', '-', 'yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was', 'first', 'released', 'in', '1967', '.', 'i', 'also', 'heard', 'that', 'at', 'first', 'it', 'was', 'seized', 'by', 'u', '.', 's', '.', 'customs', 'if', 'it', 'ever', 'tried', 'to', 'enter', 'this', 'country', ',', 'therefore', 'being', 'a', 'fan', 'of', 'films', 'considered', '"', 'controversial', '"', 'i', 'really', 'had', 'to', 'see', 'this', 'for', 'myself', '.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'the', 'plot', 'is', 'centered', 'around', 'a', 'young', 'swedish', 'drama', 'student', 'named', 'lena', 'who', 'wants', 'to', 'learn', 'everything', 'she', 'can', 'about', 'life', '.', 'in', 'particular', 'she', 'wants', 'to', 'focus', 'her', 'attention', '##s', 'to', 'making', 'some', 'sort', 'of', 'documentary', 'on', 'what', 'the', 'average', 'sw'